# Lab 9.11 &mdash; Assemble the Insight Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Bind read-only tools into an agent with create_agent
- Withhold any trade/advice tool so it cannot act
- Wrap the insight as needs_review, with its tool trace

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-11"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Now assemble the insight agent (deck slides 7&ndash;9): `@tool` **read-only** tools
(`extract_figure`, `compute`) bound with **`create_agent`** (a runnable `CompiledStateGraph`). The
agent grounds a figure and computes a metric. The design choice: the tools list is **read-only**
&mdash; `place_trade` is **not** bound &mdash; and the output is wrapped **`needs_review`** for a
human analyst. The guardrail is what's **missing** from the tools list.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

from langchain_core.tools import tool

@tool
def extract_figure(name: str) -> dict:
    """Pull a reported figure and its source from the filing."""
    return REPORT.get(name, {})
@tool
def compute(expression: str) -> float:
    """Do exact arithmetic on a financial expression."""
    return safe_calc(expression)
@tool
def place_trade(ticker: str, shares: int) -> str:
    """Place a trade. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "TRADED"
print("read-only tools:", extract_figure.name, "&", compute.name, "| withheld:", place_trade.name)

## Your Turn
Build the agent with **read-only** tools (no `place_trade`), and wrap whatever it finds as
**`needs_review`**. The insight text comes from the model at run time (the optional cell), so the
graded steps check the **wiring and the guardrail**, not the prose.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def readonly_tools():
    return ___   # TODO: read-only -- [extract_figure, compute], NEVER place_trade

def make_insight_agent():
    model = ChatOllama(model="llama3.2:1b")
    return create_agent(model, ___)   # TODO: bind the read-only tools

def analyze_report(insight, tools_used):
    # analysis only: wrap the finding so a human analyst reviews it -- never a decision
    return {"insight": insight, "status": ___, "tools_used": tools_used}   # TODO: needs_review

try:
    print('bound tools :', [t.name for t in readonly_tools()])
    print('can it trade?:', 'place_trade' in [t.name for t in readonly_tools()])
    print('agent type  :', type(make_insight_agent()).__name__)
    demo = analyze_report("Revenue 120.0M [p4, income stmt], +12.0% YoY.", ["extract_figure", "compute"])
    print('wrapped     :', demo['status'], '| tools:', demo['tools_used'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the agent is a runnable CompiledStateGraph", lambda: type(make_insight_agent()).__name__ == "CompiledStateGraph")
expect_true("it binds exactly the two read-only tools", lambda: [t.name for t in readonly_tools()] == ["extract_figure", "compute"])
expect_true("the trade tool is WITHHELD (the guardrail)", lambda: "place_trade" not in [t.name for t in readonly_tools()])
expect_true("place_trade still EXISTS as a capability, just unbound", lambda: place_trade.name == "place_trade")
expect_true("an insight is wrapped as needs_review, never a decision", lambda: analyze_report("i", [])["status"] == "needs_review")
expect_true("the wrapper preserves the tool trace", lambda: analyze_report("i", ["extract_figure"])["tools_used"] == ["extract_figure"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the read-only agent for real: it can ground & compute, but it has no way to trade. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
from langchain_core.messages import AIMessage
def tools_used(messages):
    return [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])]
try:
    if ollama_up():
        agent = make_insight_agent()
        result = agent.invoke({"messages": [{"role": "user",
                 "content": "Use extract_figure to get revenue, then state it with its source. Give NO advice."}]},
                 config={"recursion_limit": 8})
        insight = result["messages"][-1].content
        out = analyze_report(insight, tools_used(result["messages"]))
        print("tools used:", out["tools_used"])
        print("status    :", out["status"], "(the agent has no trade tool, so it cannot act)")
        print("insight   :", out["insight"])
    else:
        print("No Ollama reachable -- skipping the live run. The read-only wiring above is what matters:")
        print("place_trade is defined but never bound, so the agent grounds & computes but cannot trade.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
The guardrail is what's MISSING from the tools list -- place_trade is never bound, so the agent grounds, computes and cites but cannot trade or advise. Next: run it over a suite.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>